In [3]:
!pip install timm torch torchvision numpy tqdm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\user\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import timm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
DATASET_PATH = r"Dataset-Preprocessed"

TRAIN_PATH = os.path.join(DATASET_PATH, "Train")
VAL_PATH   = os.path.join(DATASET_PATH, "Validation")
TEST_PATH  = os.path.join(DATASET_PATH, "Test")

In [4]:
model = timm.create_model("cait_s24_224", pretrained=True)
torch.save(model.state_dict(), "cait_s24_224.pth")
print("Model saved successfully")

Model saved successfully


In [5]:
model = timm.create_model("cait_s24_224", pretrained=True)

model.load_state_dict(
    torch.load("cait_s24_224.pth", map_location=device)
)

model = model.to(device)
model.eval()

Cait(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): Sequential(
    (0): LayerScaleBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): TalkingHeadAttn(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_l): Linear(in_features=8, out_features=8, bias=True)
        (proj_w): Linear(in_features=8, out_features=8, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
 

In [6]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [7]:
def extract_features(folder_path):

    X, y = [], []

    classes = {"Fake": 0, "Real": 1}

    for cls in classes:

        class_path = os.path.join(folder_path, cls)

        for img_name in tqdm(os.listdir(class_path), desc=cls):

            img_path = os.path.join(class_path, img_name)

            try:
                img = Image.open(img_path).convert("RGB")
                img = transform(img).unsqueeze(0).to(device)

                with torch.no_grad():
                    feat = model.forward_features(img)

                    # CLS token (global info)
                    cls_token = feat[:, 0, :]

                    # patch tokens (local info)
                    patch_tokens = feat[:, 1:, :]

                    patch_mean = patch_tokens.mean(dim=1)

                    # combine → 768 features
                    final_feat = torch.cat([cls_token, patch_mean], dim=1)

                X.append(final_feat.cpu().numpy().flatten())
                y.append(classes[cls])

            except:
                continue

    return np.array(X), np.array(y)

In [8]:
SAVE_DIR = r"Features-extracted-t"
os.makedirs(SAVE_DIR, exist_ok=True)

In [20]:
X_test, y_test = extract_features(TEST_PATH)
print("Test shape:", X_test.shape)

Real: 100%|██████████| 5413/5413 [09:11<00:00,  9.82it/s]  

Test shape: (10905, 768)


In [22]:
np.save(os.path.join(SAVE_DIR, "X_test.npy"), X_test)
np.save(os.path.join(SAVE_DIR, "y_test.npy"), y_test)

In [23]:
X_train, y_train = extract_features(TRAIN_PATH)
print("Train shape:", X_train.shape)

Real: 100%|██████████| 70001/70001 [2:04:32<00:00,  9.37it/s]  


Train shape: (140002, 768)


In [24]:
np.save(os.path.join(SAVE_DIR, "X_train.npy"), X_train)
np.save(os.path.join(SAVE_DIR, "y_train.npy"), y_train)



In [ ]:
X_v, y_v = extract_features(VAL_PATH)
print("VAL shape:", X_v.shape)

Real:   8%|▊         | 1560/19787 [01:02<07:05, 42.88it/s]